# 02. Cleaned — Silver Layer

This notebook implements Stage 2 of the HDB ETL pipeline: Cleaning and data quality enforcement.

Steps applied in sequence:
1. Read from the Bronze layer.
2. Detect and convert integer date columns to DateType.
3. Drop remaining_lease, which is an unreliable source column that will be recomputed.
4. Deduplicate records using a composite key of all columns except resale_price.
5. Profile data at the column level to understand distribution and quality before validation.
6. Validate five fields against business rules: month, town, flat_type, flat_model, and storey_range.
7. Recompute remaining_lease_recomputed from lease_commence_date using the 99-year HDB lease constant.
8. Detect price anomalies using Z-Score per town and flat_type group.
9. Consolidate all rejected records into a single failed output with a reason column.
10. Write Cleaned to Silver and Failed to Silver.

Records that fail deduplication, validation, or price anomaly checks are routed to the Failed output. Only records that pass all three checks form the Cleaned output.

## Inherit Configs

In [0]:
%run "./00_nb_configs"

# 00. Configs

This notebook is the single source of truth for all pipeline-wide settings.
It is referenced by all downstream notebooks via %run 00. configs.

Contents:
- Required library imports
- SparkSession initialisation
- Schema and table name constants following the Medallion architecture (Bronze, Silver, Gold)
- Source data path
- Business logic constants (HDB lease duration and valid flat types)
- Date range filter boundaries
- Delta table creation

To customise the pipeline, only edit values in this notebook. All other notebooks inherit these values automatically.

## Required Libraries

## Source Data Path
Define where the raw HDB CSV files are located.


Source path : dbfs:/FileStore/test/


## Databricks - Catalog
- Databricks-specific catalog usage 
- If using a different cloud platform, comment out the below cell.

...

## Date Range Filter

The pipeline scope covers HDB resale transactions from January 2012 to December 2016 as per the requierment.

Date filter : 2012-01-01  to  2016-12-31


## Business Logic Constants

- HDB_LEASE is set to 99, which is the standard Singapore HDB lease duration in years. It is used in the Silver stage to compute the lease expiry date and the recomputed remaining lease string.

- VALID_FLAT_TYPES is the whitelist of officially recognised HDB flat types. Any record whose flat_type falls outside this list will be flagged as invalid during the validation step in the Silver stage.

HDB lease   : 99 years
Valid types : ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']


## Output Delta Table Creation
- Databricks-specific schema and table creation function. 
- If using a different cloud platform, comment out the schema creation line.

## Read from Bronze Layer

- Load the Bronze output produced by 01. raw as the starting point for cleaning.

- On cloud or Databricks, the Delta table bronze.raw_hdb is read directly.

- On a local machine, the parquet files written to bronze/raw_hdb are read instead.

In [0]:
cleaned_hdb_df = spark.read.table(CLD_RAW)
print(f"[CLOUD]  Read Bronze from table   : {CLD_RAW}")

print(f"Records loaded : {cleaned_hdb_df.count():,}")

[CLOUD]  Read Bronze from table   : raw.bronze.raw_hdb
Records loaded : 92,544


## Detect and Convert Integer Date Columns to DateType

- Dynamically detect all columns whose names contain the word date. For any such column that was inferred as int or bigint, meaning the source stored only a year value like 1990, append -01-01 to form a full date.

- For example, 1990 becomes the string 1990-01-01 which is then cast to a proper date.

In [0]:
date_cols = [cols for cols in cleaned_hdb_df.columns if "date" in cols.lower()]
print("Detected date columns:", date_cols)

for cols in date_cols:
    dtype = dict(cleaned_hdb_df.dtypes)[cols]
    if dtype in ["int", "bigint"]:
        cleaned_hdb_df = cleaned_hdb_df.withColumn(
            cols,
            to_date(concat(col(cols).cast("string"), lit("-01-01")))
        )
        print(f"  Converted '{cols}' from {dtype} to DateType")

Detected date columns: ['lease_commence_date']
  Converted 'lease_commence_date' from int to DateType


## Drop remaining_lease Column

- Remove the remaining_lease column before any further processing.

- This column exists in some source files but not others and its format is inconsistent across years. 

In [0]:
cleaned_hdb_df = cleaned_hdb_df.drop("remaining_lease")

print("Dropped: remaining_lease")
print("Remaining columns:", cleaned_hdb_df.columns)

Dropped: remaining_lease
Remaining columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']


## Deduplication

- Identify and remove duplicate records based on the composite key defined in the requirements, which is all columns except resale_price.

- The approach works as follows. First, key_cols is built as all column names excluding resale_price. Then a unique_id is generated per row by SHA-256 hashing the concatenation of all key column values cast to string and separated by ||. This produces a deterministic fingerprint for each transaction identity.

- A window function partitioned by unique_id and ordered by resale_price descending assigns a row_number to each record within its group. Rank 1 is the record with the highest price for that transaction identity and is kept in the Cleaned output. Any rank greater than 1 represents a lower-priced duplicate and is routed to the Failed output.

In [0]:
key_cols = [cols for cols in cleaned_hdb_df.columns if cols != "resale_price"]

cleaned_hdb_df = cleaned_hdb_df.withColumn(
    "unique_id",
    sha2(concat_ws("||", *[col(cols).cast("string") for cols in key_cols]), 256)
)

window_spec = Window.partitionBy("unique_id").orderBy(col("resale_price").desc())
ranked_hdb_df = cleaned_hdb_df.withColumn("rank", row_number().over(window_spec))

cleaned_hdb_df = ranked_hdb_df.filter(col("rank") == 1).drop("rank")

dedup_failed_df = (
    ranked_hdb_df.filter(col("rank") > 1).drop("rank")
    .withColumn("failed_reason", lit("duplicate: lower resale_price for same composite key"))
)

print(f"Records after deduplication : {cleaned_hdb_df.count():,}")
print(f"Duplicate rejects           : {dedup_failed_df.count():,}")

Records after deduplication : 90,944
Duplicate rejects           : 1,600


## Data Profiling

- Perform column-level profiling on the deduplicated dataset to understand data distribution and quality before applying validation rules.

- For each column the following metrics are computed: data_type which is the Spark-inferred column type, total_rows which is the total number of rows in the dataset, null_count which is the number of NULL values, null_percentage which is the proportion of NULLs expressed as a percentage, distinct_count which is the number of unique non-null values, and min_value and max_value which are the minimum and maximum observed values.

- The result is stored as profile_summary_df which can be displayed or written for reporting.

In [0]:
profiling_results = []
total_rows = cleaned_hdb_df.count()
print(f"Total records after deduplication: {total_rows:,}")

for column in cleaned_hdb_df.columns:
    dtype = dict(cleaned_hdb_df.dtypes)[column]
    stats = cleaned_hdb_df.select(
        count(when(col(column).isNull(), column)).alias("null_count"),
        countDistinct(column).alias("distinct_count"),
        min(column).alias("min_value"),
        max(column).alias("max_value")
    ).collect()[0]

    profiling_results.append((
        column,
        dtype,
        total_rows,
        stats["null_count"],
        (stats["null_count"] / total_rows) * 100,
        stats["distinct_count"],
        str(stats["min_value"]),
        str(stats["max_value"])
    ))

profiling_schema = [
    "column_name", "data_type", "total_rows", "null_count",
    "null_percentage", "distinct_count", "min_value", "max_value"
]

profile_summary_df = spark.createDataFrame(profiling_results, profiling_schema)
profile_summary_df.show(truncate=False)

Total records after deduplication: 90,944
+-------------------+---------+----------+----------+---------------+--------------+----------------------------------------------------------------+----------------------------------------------------------------+
|column_name        |data_type|total_rows|null_count|null_percentage|distinct_count|min_value                                                       |max_value                                                       |
+-------------------+---------+----------+----------+---------------+--------------+----------------------------------------------------------------+----------------------------------------------------------------+
|month              |date     |90944     |0         |0.0            |60            |2012-01-01                                                      |2016-12-01                                                      |
|town               |string   |90944     |0         |0.0            |26            |ANG MO KIO    

## Data Validation

- Apply business rule validations across five fields specified in the requirements: month, town, flat_type, flat_model, and storey_range.

- The rules are as follows. The month field must not be null and must not be a future date. The town field must not be null and must contain only uppercase letters and spaces matching the pattern ^[A-Z ]+$. The flat_type field must be one of the values in the VALID_FLAT_TYPES whitelist. The flat_model field must not be null. The storey_range field must match the format NN TO NN such as 01 TO 03.

- All violations for a record are concatenated into a comments column separated by semicolons. An is_valid flag is then derived where 1 means the record passed all checks and 0 means it failed at least one. Records with is_valid equal to 0 are routed to the Failed output.

In [0]:
cleaned_hdb_df = cleaned_hdb_df.withColumn(
    "comments",
    concat_ws("; ",
        when(col("month").isNull(), "month is null"),
        when(col("month") > current_date(), "future month"),
        when(col("town").isNull(), "town is null"),
        when(~col("town").rlike("^[A-Z ]+$"), "invalid town format"),
        when(~col("flat_type").isin(VALID_FLAT_TYPES), "invalid flat_type"),
        when(col("flat_model").isNull(), "flat_model is null"),
        when(~col("storey_range").rlike("^[0-9]{2} TO [0-9]{2}$"), "invalid storey_range")
    )
).withColumn(
    "is_valid",
    when(col("comments") == "", 1).otherwise(0)
)

validation_failed_df = (
    cleaned_hdb_df.filter(col("is_valid") == 0)
    .withColumn("failed_reason", concat(lit("validation failed: "), col("comments")))
    .drop("comments", "is_valid")
)

cleaned_hdb_df = cleaned_hdb_df.filter(col("is_valid") == 1).drop("comments", "is_valid")

print(f"Validation passed : {cleaned_hdb_df.count():,}")
print(f"Validation failed : {validation_failed_df.count():,}")

Validation passed : 88,256
Validation failed : 2,688


## Recompute Remaining Lease

- Recalculate the **remaining lease** for each record using **lease_commence_date** and the standard **99-year HDB lease**.

- Steps:
  - Compute **lease_expiry_date** = lease_commence_date + 99 years using add_months.
  - Calculate **remaining_months** between lease_expiry_date and **today** using months_between, rounded down to whole months.
  - Derive **remaining_years** = remaining_months / 12.
  - Derive **remaining_months_only** = remaining_months % 12.

- The final column **remaining_lease_recomputed** is formatted like **"62 years 1 month"**.

- All temporary calculation columns are **dropped after creating the final column** to keep the dataset clean.

In [0]:
cleaned_hdb_df = (
    cleaned_hdb_df
    .withColumn("lease_expiry_date",
        add_months(col("lease_commence_date"), HDB_LEASE * 12))
    .withColumn("remaining_months",
        floor(months_between(col("lease_expiry_date"), current_date())))
    .withColumn("remaining_years",
        floor(col("remaining_months") / 12))
    .withColumn("remaining_months_only",
        col("remaining_months") % 12)
    .withColumn("remaining_lease_recomputed",
        concat(
            col("remaining_years"), lit(" years "),
            col("remaining_months_only"), lit(" months")
        ))
    .drop("lease_expiry_date", "remaining_months", "remaining_years", "remaining_months_only")
)

print("remaining_lease_recomputed added.")
cleaned_hdb_df.select("lease_commence_date", "remaining_lease_recomputed").show(5, truncate=False)

remaining_lease_recomputed added.
+-------------------+--------------------------+
|lease_commence_date|remaining_lease_recomputed|
+-------------------+--------------------------+
|1998-01-01         |70 years 9 months         |
|2002-01-01         |74 years 9 months         |
|1984-01-01         |56 years 9 months         |
|2001-01-01         |73 years 9 months         |
|2013-01-01         |85 years 9 months         |
+-------------------+--------------------------+
only showing top 5 rows


## Z-Score Based Price Anomaly Detection

- Detect unusual resale prices using the **Z-Score method** within each **town and flat_type** group.

- Prices vary across towns and flat types, so grouping ensures comparisons are made only among similar properties.  
  The **Z-Score** is calculated as:

  **(resale_price − group_mean) / group_stddev**

- If the **absolute Z-Score > 3**, the record is flagged as **ANOMALY**, since values beyond 3 standard deviations are rare.

- **PySpark stddev()** uses sample standard deviation.  
  If a group has only one record, the standard deviation becomes **null**, so the record is marked **VALID** to avoid false positives.

- A **left join** ensures all records are retained even if group statistics are missing.

- Temporary columns used for Z-Score calculation are **dropped after processing** to keep the dataset clean.

In [0]:
price_stats_df = cleaned_hdb_df.groupBy("town", "flat_type").agg(
    mean("resale_price").alias("mean_price"),
    stddev("resale_price").alias("stddev_price")
)

cleaned_hdb_df = (
    cleaned_hdb_df
    .join(price_stats_df, on=["town", "flat_type"], how="left")
    .withColumn(
        "z_score",
        (col("resale_price") - col("mean_price")) / col("stddev_price")
    )
    .withColumn(
        "price_anomaly_flag",
        when(
            col("z_score").isNull() | (abs(col("z_score")) <= 3),
            "VALID"
        ).otherwise("ANOMALY")
    )
    .drop("mean_price", "stddev_price", "z_score")
)

price_failed_df = (
    cleaned_hdb_df.filter(col("price_anomaly_flag") == "ANOMALY")
    .withColumn("failed_reason", lit("price anomaly: resale_price z-score > 3 for town+flat_type"))
    .drop("price_anomaly_flag")
)

cleaned_hdb_df = (
    cleaned_hdb_df.filter(col("price_anomaly_flag") == "VALID")
    .drop("price_anomaly_flag")
)

print(f"Price check passed : {cleaned_hdb_df.count():,}")
print(f"Price anomalies    : {price_failed_df.count():,}")

Price check passed : 87,832
Price anomalies    : 424


## Consolidate Failed Records

- Union all three sources of rejected records into a single failed_records_df with a failed_reason column explaining why each record was discarded.

- dedup_failed_df carries the reason: duplicate: lower resale_price for same composite key.

- validation_failed_df carries the reason: validation failed followed by the specific rule violations.
 
- price_failed_df carries the reason: price anomaly: resale_price z-score > 3 for town+flat_type.
 
- unionByName with allowMissingColumns=True is used to safely combine DataFrames that may have slightly different column sets.

In [0]:
failed_records_df = (
    dedup_failed_df
    .unionByName(validation_failed_df, allowMissingColumns=True)
    .unionByName(price_failed_df, allowMissingColumns=True)
)

print(f"Total failed records  : {failed_records_df.count():,}")
print(f"Total cleaned records : {cleaned_hdb_df.count():,}")

Total failed records  : 4,712
Total cleaned records : 87,832


## Write Silver Outputs

- Persist both Silver layer outputs: the Cleaned records and the Failed records.

- Each DataFrame is saved as a table using saveAsTable.
  
- The Cleaned output at silver.cleaned_hdb contains all records that passed deduplication, validation, and price anomaly checks, and includes the recomputed remaining_lease_recomputed column.
 
- The Failed output at silver.failed_hdb contains all rejected records from any stage with the failed_reason column for traceability.

In [0]:
write_layer(cleaned_hdb_df,    CLD_CLEANED, "CLEANED")
write_layer(failed_records_df, CLD_FAILED,  "FAILED ")

CLEANED written to table : raw.bronze.cleaned_hdb
FAILED  written to table : raw.bronze.failed_hdb
